# 05 Models — PyTorch — Men's

Neural network trained with **Brier loss directly** (not BCE), as recommended by research. This should produce better-calibrated probabilities for the Brier-scored competition.

**Architecture**: Simple feedforward network (2 hidden layers) with dropout regularization. The limited training data (~2500 games) means we need a small network to avoid overfitting.

**Inputs** (from S3 `04_preprocessing/mens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/pytorch/mens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
except:
    ! pip install torch
    import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

PyTorch version: 2.6.0+cu124
Device: cpu


#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

In [3]:
class BrierNet(nn.Module):
    """Simple feedforward network for binary probability prediction."""
    def __init__(self, n_features, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

In [4]:
def brier_loss(preds, targets):
    """Brier score as a differentiable loss function."""
    return torch.mean((preds - targets) ** 2)

def train_model(X_train, y_train, X_val, y_val, n_features, epochs=200, lr=0.001, batch_size=128, patience=20):
    """Train a BrierNet with early stopping."""
    model = BrierNet(n_features).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    train_ds = TensorDataset(
        torch.FloatTensor(X_train).to(device),
        torch.FloatTensor(y_train).to(device)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.FloatTensor(y_val).to(device)
    
    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    best_epoch = 0
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = brier_loss(preds, y_batch)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t)
            val_loss = brier_loss(val_preds, y_val_t).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1
        
        if no_improve >= patience:
            break
    
    model.load_state_dict(best_state)
    model.to(device)
    return model, best_epoch, best_val_loss

def predict(model, X):
    """Generate predictions from a trained model."""
    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X).to(device)
        preds = model(X_t).cpu().numpy()
    return preds

#### Constants

In [ ]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "05_models/pytorch"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

#### Make output directory

In [6]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [7]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (5170, 82)
Stage 1 grid: (261013, 80)
Stage 2 grid: (66430, 79)
Features: 27


## 2. LOGO Cross-Validation

Neural nets require imputed features and standardized inputs. We fit the scaler on training folds only to avoid leakage.

In [8]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train_raw = train.loc[train_mask, feature_cols].values
    y_train_raw = train.loc[train_mask, 'Label'].values
    X_val_raw = train_original.loc[val_mask, feature_cols].values
    y_val_raw = train_original.loc[val_mask, 'Label'].values
    
    if len(X_val_raw) == 0:
        continue
    
    # Impute NaN with 0 (difference features — 0 means no difference)
    X_train_imp = np.nan_to_num(X_train_raw, nan=0.0)
    X_val_imp = np.nan_to_num(X_val_raw, nan=0.0)
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)
    
    model, best_epoch, best_val_loss = train_model(
        X_train_scaled, y_train_raw, X_val_scaled, y_val_raw,
        n_features=len(feature_cols)
    )
    
    preds = predict(model, X_val_scaled)
    brier = brier_score_loss(y_val_raw, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val_raw),
        'BestEpoch': best_epoch
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val_raw)}, BestEpoch={best_epoch}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

  Fold 1985: Brier=0.1804, Games=63, BestEpoch=47
  Fold 1986: Brier=0.2032, Games=63, BestEpoch=4
  Fold 1987: Brier=0.1787, Games=63, BestEpoch=32
  Fold 1988: Brier=0.1976, Games=63, BestEpoch=8
  Fold 1989: Brier=0.1624, Games=63, BestEpoch=49
  Fold 1990: Brier=0.1859, Games=63, BestEpoch=79
  Fold 1991: Brier=0.1865, Games=63, BestEpoch=73
  Fold 1992: Brier=0.1667, Games=63, BestEpoch=12
  Fold 1993: Brier=0.1617, Games=63, BestEpoch=33
  Fold 1994: Brier=0.1651, Games=63, BestEpoch=24
  Fold 1995: Brier=0.1690, Games=63, BestEpoch=11
  Fold 1996: Brier=0.1581, Games=63, BestEpoch=75
  Fold 1997: Brier=0.1949, Games=63, BestEpoch=4
  Fold 1998: Brier=0.1952, Games=63, BestEpoch=3
  Fold 1999: Brier=0.1825, Games=63, BestEpoch=109
  Fold 2000: Brier=0.1697, Games=63, BestEpoch=39
  Fold 2001: Brier=0.2041, Games=64, BestEpoch=1
  Fold 2002: Brier=0.1947, Games=64, BestEpoch=18
  Fold 2003: Brier=0.1681, Games=64, BestEpoch=54
  Fold 2004: Brier=0.1759, Games=64, BestEpoch=1
  Fol

In [9]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")


Overall OOF Brier Score: 0.1827
Stage 1 (2022-2025) Brier Score: 0.1892
Mean Brier: 0.1826 +/- 0.0197


## 3. Train Final Model and Calibrate

In [10]:
# Fit scaler on all training data
X_all = np.nan_to_num(train[feature_cols].values, nan=0.0)
y_all = train['Label'].values

final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X_all)

# Use a small held-out portion for early stopping in final model
n_val = int(len(X_all_scaled) * 0.1)
indices = np.random.permutation(len(X_all_scaled))
X_final_train = X_all_scaled[indices[n_val:]]
y_final_train = y_all[indices[n_val:]]
X_final_val = X_all_scaled[indices[:n_val]]
y_final_val = y_all[indices[:n_val]]

final_model, best_epoch, _ = train_model(
    X_final_train, y_final_train, X_final_val, y_final_val,
    n_features=len(feature_cols), epochs=300
)
print(f"Final model best epoch: {best_epoch}")

# Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

Final model best epoch: 8
OOF Brier (raw): 0.1827
OOF Brier (calibrated): 0.1797


## 4. Generate Predictions

In [11]:
X_stage1 = final_scaler.transform(np.nan_to_num(stage1[feature_cols].values, nan=0.0))
stage1['Pred_pytorch'] = predict(final_model, X_stage1)
stage1['Pred_pytorch_calibrated'] = calibrator.predict(stage1['Pred_pytorch'].values).clip(0.02, 0.98)

X_stage2 = final_scaler.transform(np.nan_to_num(stage2[feature_cols].values, nan=0.0))
stage2['Pred_pytorch'] = predict(final_model, X_stage2)
stage2['Pred_pytorch_calibrated'] = calibrator.predict(stage2['Pred_pytorch'].values).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_pytorch_calibrated'].min():.3f}, {stage1['Pred_pytorch_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_pytorch_calibrated'].min():.3f}, {stage2['Pred_pytorch_calibrated'].max():.3f}]")

stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_pytorch_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

Stage 1 predictions range: [0.020, 0.980]
Stage 2 predictions range: [0.020, 0.980]
Stage 1 Brier (calibrated): 0.1876


## 5. Save Outputs

In [12]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_pytorch', 'Pred_pytorch_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_pytorch', 'Pred_pytorch_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/pytorch/mens/oof_predictions.parquet ((2585, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/pytorch/mens/stage1_predictions.parquet ((261013, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/pytorch/mens/stage2_predictions.parquet ((66430, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/pytorch/mens/cv_results.parquet ((40, 4))


## 6. Summary

In [13]:
print("=" * 60)
print("PYTORCH MODEL SUMMARY — MEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
print(f"CV Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Architecture: BrierNet(64 -> 32 -> 1) with Brier loss")
print(f"Loss function: Brier (MSE of probabilities)")

PYTORCH MODEL SUMMARY — MEN'S

OOF Brier Score (raw): 0.1827
OOF Brier Score (calibrated): 0.1797
CV Mean Brier: 0.1826 +/- 0.0197
Architecture: BrierNet(64 -> 32 -> 1) with Brier loss
Loss function: Brier (MSE of probabilities)
